In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [3]:
# Paths
BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TRAIN_PATH = os.path.join(BASE_PATH, "genres_stems")
MASHUP_PATH = os.path.join(BASE_PATH, "mashups")
ESC50_PATH = os.path.join(BASE_PATH, "ESC-50-master", "audio")

In [4]:
# Explore train stems
durations, sample_rates, classes = [], [], []

for genre in os.listdir(TRAIN_PATH):
    genre_path = os.path.join(TRAIN_PATH, genre)
    classes.append(genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        for stem in ["drums.wav","bass.wav","vocals.wav","other.wav"]:
            file_path = os.path.join(song_path, stem)
            y, sr = librosa.load(file_path, sr=None)
            durations.append(len(y)/sr)
            sample_rates.append(sr)

print("Total files:", len(durations))
print("Unique sample rates:", set(sample_rates))
print("Average duration:", np.mean(durations))
print("Class distribution:", {g:len(os.listdir(os.path.join(TRAIN_PATH,g))) for g in classes})

Total files: 4000
Unique sample rates: {44100}
Average duration: 30.02404707482993
Class distribution: {'disco': 100, 'metal': 100, 'reggae': 100, 'blues': 100, 'rock': 100, 'classical': 100, 'jazz': 100, 'hiphop': 100, 'country': 100, 'pop': 100}


In [5]:
genres = sorted(os.listdir(TRAIN_PATH))
label_map = {g:i for i,g in enumerate(genres)}
rev_label_map = {v:k for k,v in label_map.items()}

data = []
for genre in genres:
    genre_path = os.path.join(TRAIN_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        stems = {
            "drums": os.path.join(song_path,"drums.wav"),
            "bass": os.path.join(song_path,"bass.wav"),
            "vocals": os.path.join(song_path,"vocals.wav"),
            "other": os.path.join(song_path,"other.wav")
        }
        data.append((stems,label_map[genre]))

print("Total songs:", len(data))

Total songs: 1000


In [6]:
train_data, val_data = train_test_split(
    data,
    test_size=0.1,
    random_state=42,
    stratify=[x[1] for x in data]
)

In [7]:
class MashupDataset(Dataset):
    def __init__(self, data, esc50_path=None, augment=True):
        self.data = data
        self.target_len = 22050 * 30
        self.augment = augment
        self.esc50_files = []

        if esc50_path:
            for f in os.listdir(esc50_path):
                if f.endswith(".wav"):
                    self.esc50_files.append(os.path.join(esc50_path, f))

        # group data by label (IMPORTANT for cross mixing)
        self.class_data = {}
        for stems, label in data:
            if label not in self.class_data:
                self.class_data[label] = []
            self.class_data[label].append(stems)

    def fix_length(self, y):
        if len(y) < self.target_len:
            y = np.pad(y, (0, self.target_len - len(y)))
        else:
            y = y[:self.target_len]
        return y

    def __len__(self):
        return len(self.data)

    def load_audio(self, path):
        y, _ = librosa.load(path, sr=22050)
        return self.fix_length(y)

    def __getitem__(self, idx):
        stems, label = self.data[idx]

        audio = []

        # 🔥 CROSS-SONG STEM MIXING
        for key in ["drums", "bass", "vocals", "other"]:
            if self.augment and random.random() < 0.7:
                # pick random song from SAME GENRE
                rand_stem = random.choice(self.class_data[label])
                path = rand_stem[key]
            else:
                path = stems[key]

            y = self.load_audio(path)

            # 🔥 RANDOM VOLUME SCALING
            if self.augment:
                gain = random.uniform(0.5, 1.5)
                y = y * gain

            # 🔥 RANDOM STEM DROPOUT
            if self.augment and random.random() < 0.2:
                y = np.zeros_like(y)

            audio.append(y)

        mix = np.sum(audio, axis=0)

        # 🔥 TEMPO AUGMENTATION
        if self.augment and random.random() < 0.5:
            rate = random.uniform(0.8, 1.2)
            mix = librosa.effects.time_stretch(mix, rate=rate)
            mix = self.fix_length(mix)

        # 🔥 STRONG NOISE (MULTIPLE + RANDOM POSITION)
        if self.augment and len(self.esc50_files) > 0:
            for _ in range(random.randint(1, 3)):
                noise_file = random.choice(self.esc50_files)
                noise, _ = librosa.load(noise_file, sr=22050)

                start = random.randint(0, max(0, len(mix) - len(noise)))
                noise_pad = np.zeros_like(mix)
                end = min(len(mix), start + len(noise))
                noise_pad[start:end] = noise[:end-start]

                # stronger noise
                intensity = random.uniform(0.3, 1.0)
                mix = mix + intensity * noise_pad

        # 🔥 MEL SPECTROGRAM
        mel = librosa.feature.melspectrogram(y=mix, sr=22050, n_mels=128)
        mel = librosa.power_to_db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        mel = torch.tensor(mel).unsqueeze(0).float()

        return mel, label

In [8]:
train_dataset = MashupDataset(train_data, esc50_path=ESC50_PATH, augment=True)
val_dataset   = MashupDataset(val_data, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)

In [9]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),  # deeper start
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Linear(128, 10)  # 10 genres

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 25

In [11]:
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Validation
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)
            p = torch.argmax(out, 1).cpu().numpy()
            preds.extend(p)
            targets.extend(y.numpy())
    
    f1 = f1_score(targets, preds, average="macro")
    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | Val Macro F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best model!")

Epoch 1 | Loss: 2.1104 | Val Macro F1: 0.2164
Saved best model!
Epoch 2 | Loss: 1.9447 | Val Macro F1: 0.3214
Saved best model!
Epoch 3 | Loss: 1.8559 | Val Macro F1: 0.3504
Saved best model!
Epoch 4 | Loss: 1.7627 | Val Macro F1: 0.3425
Epoch 5 | Loss: 1.6988 | Val Macro F1: 0.3265
Epoch 6 | Loss: 1.6410 | Val Macro F1: 0.2658
Epoch 7 | Loss: 1.5761 | Val Macro F1: 0.4902
Saved best model!
Epoch 8 | Loss: 1.5790 | Val Macro F1: 0.3975
Epoch 9 | Loss: 1.5934 | Val Macro F1: 0.2829
Epoch 10 | Loss: 1.5884 | Val Macro F1: 0.4752
Epoch 11 | Loss: 1.5501 | Val Macro F1: 0.2467
Epoch 12 | Loss: 1.5027 | Val Macro F1: 0.3152
Epoch 13 | Loss: 1.5130 | Val Macro F1: 0.4345
Epoch 14 | Loss: 1.5043 | Val Macro F1: 0.2620
Epoch 15 | Loss: 1.4341 | Val Macro F1: 0.3070
Epoch 16 | Loss: 1.4211 | Val Macro F1: 0.3315
Epoch 17 | Loss: 1.4335 | Val Macro F1: 0.1781
Epoch 18 | Loss: 1.4518 | Val Macro F1: 0.4455
Epoch 19 | Loss: 1.4139 | Val Macro F1: 0.4959
Saved best model!
Epoch 20 | Loss: 1.3747 | 

In [12]:
class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.target_len = 22050*30

    def fix_length(self, y):
        if len(y) < self.target_len:
            y = np.pad(y,(0,self.target_len-len(y)))
        else:
            y = y[:self.target_len]
        return y

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = os.path.join(BASE_PATH, self.df.iloc[idx]["filename"])
        y, sr = librosa.load(path, sr=22050)
        y = self.fix_length(y)

        # ✅ fixed
        mel = librosa.feature.melspectrogram(y=y, sr=22050, n_mels=128)

        mel = librosa.power_to_db(mel)
        mel = (mel - mel.mean())/(mel.std()+1e-6)
        mel = torch.tensor(mel).unsqueeze(0).float()
        return mel

test_df = pd.read_csv(os.path.join(BASE_PATH,"test.csv"))
test_loader = DataLoader(TestDataset(test_df), batch_size=8, shuffle=False)

In [13]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

predictions = []

with torch.no_grad():
    for x in test_loader:
        x = x.to(device)
        out = model(x)
        pred = torch.argmax(out,1).cpu().numpy()
        predictions.extend(pred)

genres_pred = [rev_label_map[p] for p in predictions]

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": genres_pred
})

submission.to_csv("submission.csv", index=False)
print(submission.head())

   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
